# 08 · Clean missing data and save your work

**Level:** Intermediate · **Practice time:** 30–50 minutes

[← Previous](07_randomness_and_simulation.ipynb) · [Course guide](../README.md) · [Next →](09_sorting_searching_grouping.ipynb)

**You will learn:** Detect invalid values, impute deliberately, and round-trip text and binary data.

**Prerequisites:** Notebooks 00–07.

## How to learn with this notebook

1. Run cells from top to bottom. Predict results before executing each example.
2. Do the **Try it** experiment by editing the example or adding a cell below it.
3. Replace `None` in each exercise with your answer, then rerun its check cell.
4. Consult the solutions at the end only after trying. An untouched exercise prints **Pending**, so Run All succeeds without claiming you solved it.
5. Complete the independent challenge, then restart the kernel and run all cells again.

Each notebook imports its own dependencies and creates its own data. You do not need variables from another notebook. The supplied outputs show a verified reference run; rerun cells to explore your changes.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## Step 1 · Missing is not zero

NaN represents a missing or undefined floating point value. Test it with `isnan`, because NaN does not equal itself. `isfinite` rejects both NaN and infinities. Here we treat infinity as invalid and convert it to NaN; this is a data policy, not a universal rule.

In [2]:
raw = np.array([[10., np.nan], [12., 20.], [np.inf, 22.]])
cleaned = raw.copy()
cleaned[~np.isfinite(cleaned)] = np.nan
print("Missing mask:\n", np.isnan(cleaned))
print("Column means ignoring missing:", np.nanmean(cleaned, axis=0))
assert np.isnan(cleaned).sum() == 2

Missing mask:
 [[False  True]
 [False False]
 [ True False]]
Column means ignoring missing: [11. 21.]


**Try it:** Count missing entries by row and by column.

## Step 2 · Imputation with a stated assumption

Imputation replaces missing values. Column-mean imputation preserves the observed column mean but changes the distribution and can understate variation. An entirely missing column needs another policy; here we reject it before computing its mean.

In [3]:
if np.any(np.all(np.isnan(cleaned), axis=0)):
    raise ValueError("A column has no observed values")
column_means = np.nanmean(cleaned, axis=0)
imputed = np.where(np.isnan(cleaned), column_means, cleaned)
print(imputed)
assert np.isfinite(imputed).all()
np.testing.assert_allclose(imputed.mean(axis=0), column_means)

[[10. 21.]
 [12. 20.]
 [11. 22.]]


**Try it:** Compare the standard deviation before and after filling a missing entry.

## Step 3 · CSV and NumPy binary formats

CSV is readable but does not preserve array metadata as reliably as `.npy`. Use `.npz` for multiple named arrays. `genfromtxt` can parse simple numeric text with missing entries. Temporary files make this example safe to rerun; replace the directory with a chosen Path when you want persistent files.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
from io import StringIO
parsed = np.genfromtxt(StringIO("temperature,humidity\n20,50\n22,\n"), delimiter=",", skip_header=1)
print("Parsed CSV:\n", parsed)
with TemporaryDirectory() as folder:
    folder = Path(folder)
    np.savetxt(folder / "clean.csv", imputed, delimiter=",", header="a,b", comments="")
    from_csv = np.loadtxt(folder / "clean.csv", delimiter=",", skiprows=1)
    np.save(folder / "clean.npy", imputed)
    from_npy = np.load(folder / "clean.npy", allow_pickle=False)
    np.savez_compressed(folder / "bundle.npz", data=imputed, means=column_means)
    with np.load(folder / "bundle.npz", allow_pickle=False) as archive:
        print("Saved names:", archive.files)
        np.testing.assert_allclose(archive["data"], imputed)
    np.testing.assert_allclose(from_csv, imputed)
    np.testing.assert_array_equal(from_npy, imputed)

Parsed CSV:
 [[20. 50.]
 [22. nan]]
Saved names: ['data', 'means']


**Try it:** Save an integer array as .npy and confirm that its dtype survives loading.

## Your practice exercises

Use the named answer variables. The checks verify your result; they do not require the same approach as the reference solution. If an assertion fails, read the expected value and inspect your shape and dtype.

### Exercise 1

Keep only rows where every value is finite.

In [5]:
measurements = np.array([[1., 2.], [np.nan, 3.], [4., 5.], [6., np.inf]])

valid_rows = None  # TODO: replace None with your answer

In [6]:
if valid_rows is None:
    print("Pending: complete Exercise 1 first.")
else:
    np.testing.assert_allclose(valid_rows, [[1, 2], [4, 5]])
    print("Exercise 1 passed!")

Pending: complete Exercise 1 first.


### Exercise 2

Replace NaN entries with the median of the observed values.

In [7]:
values = np.array([1., np.nan, 3., 9.])

filled_values = None  # TODO: replace None with your answer

In [8]:
if filled_values is None:
    print("Pending: complete Exercise 2 first.")
else:
    np.testing.assert_allclose(filled_values, [1, 3, 3, 9])
    print("Exercise 2 passed!")

Pending: complete Exercise 2 first.


## Independent challenge

Create a numeric CSV with a header and two missing values. Load it, report missing counts, apply a documented filling policy, and save a verified .npy copy.

Add code and explanation cells below. Decide on expected results before checking your implementation.

In [9]:
# Your independent challenge work goes here.

## Before you move on

- Explain the main idea in your own words without reading the examples.
- Make both exercise checks print **passed**.
- Try at least one changed input or edge case and explain the result.
- Finish the independent challenge or write down a specific question to revisit.

Expertise comes from repeated practice and debugging unfamiliar problems. Completing these notebooks gives you a broad practical foundation, not a substitute for that experience.

---

## Reference solutions — try first

These cells use separate reference variables, so they do not fill in your exercise answers. They also check the reference result. Compare your reasoning as well as the output.

### Solution 1

Keep only rows where every value is finite.

**Why it works:** isfinite builds an element mask; all(axis=1) requires every entry in a row to be valid before that row is selected.

In [10]:
reference_valid_rows = measurements[np.isfinite(measurements).all(axis=1)]
np.testing.assert_allclose(reference_valid_rows, [[1, 2], [4, 5]])
print(reference_valid_rows)

[[1. 2.]
 [4. 5.]]


### Solution 2

Replace NaN entries with the median of the observed values.

**Why it works:** nanmedian ignores NaN when estimating the replacement value. where replaces only missing entries.

In [11]:
reference_filled_values = np.where(np.isnan(values), np.nanmedian(values), values)
np.testing.assert_allclose(reference_filled_values, [1, 3, 3, 9])
print(reference_filled_values)

[1. 3. 3. 9.]


## Continue learning

- [Official NumPy documentation: how-to-io ](https://numpy.org/doc/stable/user/how-to-io.html)

[Next →](09_sorting_searching_grouping.ipynb)